In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# ETAPA 1 E 2: COLETA E CONCATENAÇÃO
# ==========================================
arquivos_crimes = glob.glob('./dados_crimes/*.csv')
dfs = []

for arquivo in arquivos_crimes:
    # Lendo cada arquivo (ajuste o sep e encoding conforme seu arquivo real)
    df_temp = pd.read_csv(arquivo, sep=';', encoding='latin1')
    
    # Padronização simples das colunas usando só métodos de string do pandas
    df_temp.columns = df_temp.columns.str.strip().str.lower().str.replace(' ', '_')
    dfs.append(df_temp)

df_crimes = pd.concat(dfs, ignore_index=True)

# ==========================================
# ETAPA 3: LIMPEZA E PADRONIZAÇÃO
# ==========================================
df_crimes = df_crimes.drop_duplicates()

# Ajuste da data (lembre de trocar 'data' pelo nome real da coluna)
df_crimes['data'] = pd.to_datetime(df_crimes['data'], errors='coerce')
df_crimes = df_crimes.dropna(subset=['data'])

# Preenchendo valores nulos de forma geral
df_crimes = df_crimes.fillna('Nao Informado')

# ==========================================
# ETAPA 4: INTEGRAÇÃO COM CLIMA
# ==========================================
df_clima = pd.read_csv('./clima.csv', sep=';')
df_clima.columns = df_clima.columns.str.strip().str.lower().str.replace(' ', '_')
df_clima['data'] = pd.to_datetime(df_clima['data'], errors='coerce')

# Criando a chave de cidade para o merge
df_crimes['cidade'] = 'Passo Fundo'
df_clima['cidade'] = 'Passo Fundo'

# Merge usando data e cidade
df_final = pd.merge(df_crimes, df_clima, on=['data', 'cidade'], how='left')

# Preenchendo nulos gerados no clima com a mediana usando só o pandas
df_final['temperatura'] = df_final['temperatura'].fillna(df_final['temperatura'].median())
df_final['chuva'] = df_final['chuva'].fillna(df_final['chuva'].median())

# ==========================================
# ETAPA 5: TRANSFORMAÇÕES E OUTLIERS
# ==========================================
# Cálculo do IQR para 'chuva' usando estatística nativa do pandas
Q1 = df_final['chuva'].quantile(0.25)
Q3 = df_final['chuva'].quantile(0.75)
IQR = Q3 - Q1
limite_sup = Q3 + 1.5 * IQR

# Criando coluna de alerta de outlier com condicional do pandas (.astype(int) transforma True/False em 1/0)
df_final['chuva_extrema'] = (df_final['chuva'] > limite_sup).astype(int)

# Normalização Min-Max manual (apenas matemática do pandas, sem sklearn)
df_final['temp_norm'] = (df_final['temperatura'] - df_final['temperatura'].min()) / (df_final['temperatura'].max() - df_final['temperatura'].min())

# ==========================================
# ETAPA 6: EXPLORAÇÃO E DESCOBERTA
# ==========================================
# Agrupando os dados por dia para poder gerar os gráficos
df_diario = df_final.groupby('data').agg(
    total_crimes=('tipo_crime', 'count'), # Conta quantas linhas (crimes) no dia
    temp_media=('temperatura', 'mean'),   # Pega a temperatura do dia
    chuva_media=('chuva', 'mean')         # Pega a chuva do dia
).reset_index()

# 1. Gráfico de linha (Série temporal)
plt.figure(figsize=(10, 4))
sns.lineplot(data=df_diario, x='data', y='total_crimes')
plt.title('Volume de Crimes ao longo do Tempo')
plt.show()

# 2. Gráfico de dispersão (Clima x Crime)
plt.figure(figsize=(10, 4))
sns.scatterplot(data=df_diario, x='temp_media', y='total_crimes')
plt.title('Relação entre Temperatura Média e Total de Crimes')
plt.show()

# 3. Matriz de Correlação
correlacao = df_diario[['total_crimes', 'temp_media', 'chuva_media']].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(correlacao, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlação: Criminalidade vs Clima')
plt.show()

# Salvando a entrega final
df_final.to_csv('dataset_final_integrado.csv', index=False)